In [42]:
# ===========================================================================
# CELL 0 — Relax binding constraints on the base LTS scenario.
# Run ONCE before Cell 1. Safe to re-run: skips already-removed constraints.
# Original values are saved to CSV in DATA_DIR for post-solve comparison.
# ===========================================================================

import ixmp
import message_ix
from pathlib import Path

DATA_DIR = Path(r"C:\Users\User\Desktop\lums\3rd semester\Thesis\Hydro-Energy Model")
mp = ixmp.Platform(backend="jdbc", driver="hsqldb", path="C:/Users/User/.local/share/ixmp/thesis_db/default")

# Constraints to remove from the base LTS scenario.
# Format: { parameter_name: [technology, ...] }
# NOTE: constraints re-introduced by water_build() are handled in Cell 1.
REMOVALS = {
    "bound_activity_up":     ["coal_imp"],
    "growth_activity_lo":    ["coal_i"],
    "initial_activity_lo":   ["coal_i"],
    "bound_activity_lo":     ["coal_i", "nuc_hc", "gas_cc", "coal_adv",
                               "gas_ppl", "coal_ppl_u", "CH4s_TCE", "CO2s_TCE", "CO2t_TCE"],
    "bound_new_capacity_up": ["wind_ppl", "nuc_hc"],
}

try:
    scen = message_ix.Scenario(mp, "MESSAGEix-Pakistan 1", "LTS", version=1)
    print(f"Loaded: {scen.model} / {scen.scenario} v{scen.version}")

    if scen.has_solution():
        scen.remove_solution()

    removed = []
    with scen.transact("Relax binding constraints for water-energy nexus"):
        for par, techs in REMOVALS.items():
            for tec in techs:
                df = scen.par(par, filters={"technology": [tec]})
                if not df.empty:
                    csv_path = DATA_DIR / f"original_{par}_{tec}.csv"
                    if not csv_path.exists():
                        df.to_csv(csv_path, index=False)
                    scen.remove_par(par, df)
                    removed.append(f"{par} [{tec}]")

    if removed:
        scen.set_as_default()
        print(f"Removed {len(removed)} constraint(s):\n  " + "\n  ".join(removed))
        print(f"\nLTS v{scen.version} set as default. Run Cell 1 next.")
    else:
        print("All constraints already removed. Run Cell 1 next.")

finally:
    mp.close_db()

Loaded: MESSAGEix-Pakistan 1 / LTS v1
All constraints already removed. Run Cell 1 next.


In [43]:
import os
import ixmp
import message_ix
import pandas as pd
from message_ix import make_df
from message_ix_models import Context
from message_ix_models.model.water.build import main as water_build
from message_ix_models.model.water.cli import water_ini

# ============================================================
# Platform + Context setup
# ============================================================
# os.environ["JAVA_TOOL_OPTIONS"] = "-Xmx2G"   # prevent OOM when loading GDX
mp = ixmp.Platform(backend="jdbc", driver="hsqldb", path="C:/Users/User/.local/share/ixmp/thesis_db/default")

context = Context()
context.handle_cli_args(
    model_name="MESSAGEix-Pakistan 1",
    scenario_name="LTS",
    # no version = loads the default set by Cell 0 (relaxed LTS)
)
context.core._mp = mp
context.update(regions="IRB")
context.type_reg = "basin"
context.map_ISO_c = {"IRB": None}
context.ssp = "SSP1"
context.RCP = "2p6"
context.SDG = "baseline"
context.REL = "low"
context.time = "year"
context.nexus_set = "nexus"
context.output_scenario = "LTS"
context.output_model = "MESSAGEix-Pakistan 1"

water_ini(context, "IRB", None)

# ============================================================
# Step 1: Clone base LTS -> LTS_nexus
# ============================================================
sc_ref = context.get_scenario()
scen = sc_ref.clone(
    model=context.output_model,
    scenario=context.output_scenario + "_nexus",
    keep_solution=False
)
print("Cloned to: {} / {} v{}".format(scen.model, scen.scenario, scen.version))

# ============================================================
# Step 2: Build water nexus data
# ============================================================
water_build(context, scen)
print("Water nexus build complete.")

# ============================================================
# Step 2b: Fix electricity supply for non-Pakistan water nodes
# Water treatment technologies at AFGHAN_SOUTH, AFGHAN_NORTH,
# CHINA, INDIA_WEST, INDIA_EAST consume electricity but those
# nodes have no supply and are not connected to R12_PAK.
# This adds a zero-cost supply so the balance can close.
# ============================================================
_EXT_NODES   = ["AFGHAN_SOUTH", "AFGHAN_NORTH", "CHINA", "INDIA_WEST", "INDIA_EAST"]
_MODEL_YEARS = [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060, 2070]
_rows = []
for _node in _EXT_NODES:
    for _yr in _MODEL_YEARS:
        _rows.append({
            "node_loc":   _node, "technology": "electr_supply_water_ext",
            "year_vtg":   _yr,   "year_act":   _yr,   "mode": "M1",
            "node_dest":  _node, "commodity":  "electr", "level": "final",
            "time": "year", "time_dest": "year", "value": 1.0, "unit": "GWa",
        })
with scen.transact("Fix: electr_supply_water_ext for non-Pakistan water basins"):
    scen.add_set("technology", "electr_supply_water_ext")
    scen.add_par("output", pd.DataFrame(_rows))
print("  [fix] electr_supply_water_ext added at {} external nodes.".format(len(_EXT_NODES)))

# ============================================================
# Step 2c: Remove bound_activity_lo constraints re-introduced
# by water_build() (cool_tech + non_cooling_tec functions).
# These technologies were relaxed on the base LTS in Cell 0,
# but water_build() injects new lower bounds for them on the
# nexus clone. foil_ppl / bio_ppl / solar_res_hist_* are new
# additions that only exist after the water module is built.
# ============================================================
POST_BUILD_REMOVALS = [
    "nuc_hc", "gas_cc", "coal_adv", "gas_ppl", "coal_ppl_u",
    "foil_ppl", "bio_ppl", "solar_res_hist_2015", "solar_res_hist_2020",
]
with scen.transact("Remove post-build bound_activity_lo from nexus scenario"):
    for _tec in POST_BUILD_REMOVALS:
        _df = scen.par("bound_activity_lo", filters={"technology": [_tec]})
        if not _df.empty:
            scen.remove_par("bound_activity_lo", _df)
            print(f"  Removed bound_activity_lo [{_tec}]")
print("Post-build constraint removal complete.")

scen.set_as_default()

# NOTE: mp is kept open intentionally -- the Post-Analysis cell (Cell 2)
# reuses this same platform connection. Run the Cleanup cell last.

[DEBUG] 22:03:06 - message_ix_models.util.common: 'water config.yaml' already loaded; skip
[DEBUG] 22:03:06 - message_ix_models.util.common: 'water set.yaml' already loaded; skip
[DEBUG] 22:03:06 - message_ix_models.util.common: 'water technology.yaml' already loaded; skip
[INFO] 22:03:06 - message_ix_models.model.water.cli: Regions choice IRB
[INFO] 22:03:09 - message_ix_models.model.water.cli: Using the following time-step for the water module: ['year']


Cloned to: MESSAGEix-Pakistan 1 / LTS_nexus v28


[INFO] 22:03:11 - message_ix_models.model.water.build: Set up MESSAGEix-Nexus
[INFO] 22:03:11 - message_ix_models.model.water.utils: Basin filtering disabled, returning all basins
[INFO] 22:03:11 - message_ix_models.model.build: Set 'mode'
[INFO] 22:03:11 - message_ix_models.model.build:   6 elements
[INFO] 22:03:11 - message_ix_models.model.build:   Check 0 required elements
[INFO] 22:03:11 - message_ix_models.model.build:   Add 48 element(s)
[DEBUG] 22:03:11 - message_ix_models.model.build:   MAFGHAN_SOUTH, MAFGHAN_NORTH, ..., MPAKISTAN, MPAKISTAN
[INFO] 22:03:11 - message_ix_models.model.build:   ---
[INFO] 22:03:11 - message_ix_models.model.build: Set 'node'
[INFO] 22:03:11 - message_ix_models.model.build:   3 elements
[INFO] 22:03:11 - message_ix_models.model.build:   Check 0 required elements
[INFO] 22:03:11 - message_ix_models.model.build:   Add 54 element(s)
[DEBUG] 22:03:11 - message_ix_models.model.build:   B1|AFGHAN_SOUTH, B2|AFGHAN_NORTH, ..., B23|PAKISTAN, B24|PAKISTAN
[IN

 future year =  [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060, 2070]
 year_wat =  (2010, 2015, 2020, 2025, 2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060, 2070)


[INFO] 22:03:23 - message_ix_models.model.water.utils: Basin filtering disabled, returning all basins
[INFO] 22:03:23 - message_ix_models.util: 6432 rows in 'input'
[DEBUG] 22:03:23 - message_ix_models.util: 
      time         node_loc         technology  year_vtg  year_act mode      node_origin           commodity              level time_origin     value     unit
0     year  B1|AFGHAN_SOUTH        return_flow      2010      2010   M1  B1|AFGHAN_SOUTH  surfacewater_basin  water_avail_basin        year  1.000000      MCM
1     year  B1|AFGHAN_SOUTH        return_flow      2015      2015   M1  B1|AFGHAN_SOUTH  surfacewater_basin  water_avail_basin        year  1.000000      MCM
...    ...              ...                ...       ...       ...  ...              ...                 ...                ...         ...       ...      ...
6430  year     B23|PAKISTAN  extract_gw_fossil      2070      2070   M1          R12_PAK              electr              final        year  0.000259  GWa/

RuntimeError: Water build failed during core apply_spec in message_ix_models.model.water.build: RuntimeError: Water build failed in add_e_flow() before add_par_data: KeyError: "['Unnamed: 0'] not found in axis"

In [ ]:
# ============================================================
# Step 3: Solve
# ============================================================
caseName = scen.model + "__" + scen.scenario + "__v" + str(scen.version)
print("\nSolving: " + caseName)
scen.solve(
    solve_options={"lpmethod": "1", "scaind": "1"},
    case=caseName
)
print("\nnexus solved successfully!")


Solving: MESSAGEix-Pakistan 1__LTS_nexus__v27


[INFO] 21:59:24 - message_ix.models: Use CPLEX options {'advind': 0, 'lpmethod': '1', 'threads': 4, 'epopt': 1e-06, 'scaind': '1'}



nexus solved successfully!


In [ ]:
import importlib
import message_ix_models.model.water.report as _water_report
importlib.reload(_water_report)  # pick up any changes made to report.py
from message_ix_models.model.water.report import report_full
from message_ix_models.util import package_data_path
from pathlib import Path

# ── Reconnect to solved scenario (safe after kernel restart) ─────────────
import ixmp
import message_ix
try:
    scen  # already defined (kernel not restarted)
except NameError:
    mp   = ixmp.Platform(backend="jdbc", driver="hsqldb",
                         path="C:/Users/User/.local/share/ixmp/thesis_db/default")
    scen = message_ix.Scenario(mp, "MESSAGEix-Pakistan 1", "LTS_nexus")
    print(f"Reconnected: {scen.model} / {scen.scenario} v{scen.version}")


In [ ]:
# ===========================================================================
# CELL 3A — REPORTING via report.py
# Calls report_full() from the water nexus report module.
# Output CSV: reporting_output/<model>_<scenario>_nexus[_vN].csv
# Re-runs produce _v1, _v2, ... so no previous result is overwritten.
# ===========================================================================


# ── Call report.py (this is the standard water nexus reporting function) ──
print(f"Reporting: {scen.model} / {scen.scenario}")

report_full(scen, reg="IRB", ssp="SSP2", sdgs=False)

# ── Find the most-recently written nexus CSV ───────────────────────────────────
import pandas as pd
out_path = package_data_path().parents[0] / "reporting_output"
_base = f"{scen.model}_{scen.scenario}_nexus"
# Collect all matching files (base + versioned) and pick the newest by mtime
_candidates = sorted(
    [f for f in out_path.glob(f"{_base}*.csv")],
    key=lambda f: f.stat().st_mtime,
    reverse=True,
)
if _candidates:
    csv_file = _candidates[0]
    df = pd.read_csv(csv_file)
    print(f"Done — CSV saved to:")
    print(f"  {csv_file}")
    print(f"  {len(df):,} rows | {df['variable'].nunique()} variables")
    print("Run Cell 3B to analyse and plot the results.")
else:
    print(f"WARNING: no nexus CSV found in {out_path}")
    print("Check the reporting_output folder manually.")

Reporting: MESSAGEix-Pakistan 1 / LTS_nexus


[INFO] 21:59:52 - message_ix_models.model.water.report: Regions given as IRB; no warranty if it's not in ['R11','R12']
[INFO] 21:59:52 - message_ix_models.model.water.report: _build_water_report_df: 194 technologies
[INFO] 22:00:06 - message_ix_models.model.water.report: _build_water_report_df: 943 variable rows built
[INFO] 22:00:06 - message_ix_models.model.water.report: CSV saved (fast path, IAMC names): C:\Users\User\Desktop\lums\3rd semester\Thesis\Hydro-Energy Model\message-ix-models\message_ix_models\reporting_output\MESSAGEix-Pakistan 1_LTS_nexus_nexus_v1.csv
[INFO] 22:00:06 - message_ix_models.model.water.report: overall reporting completed
[INFO] 22:00:06 - message_ix_models.model.water.report: Report complete → C:\Users\User\Desktop\lums\3rd semester\Thesis\Hydro-Energy Model\message-ix-models\message_ix_models\reporting_output\MESSAGEix-Pakistan 1_LTS_nexus_nexus.csv


Done — CSV saved to:
  C:\Users\User\Desktop\lums\3rd semester\Thesis\Hydro-Energy Model\message-ix-models\message_ix_models\reporting_output\MESSAGEix-Pakistan 1_LTS_nexus_nexus_v1.csv
  9,891 rows | 78 variables
Run Cell 3B to analyse and plot the results.


In [ ]:
# CLEANUP — run this last after Post-Analysis is complete
# Closes the database connection that was kept open across cells.
mp.close_db()
print("Database connection closed.")

Database connection closed.
